In [4]:
import nibabel as nib
from collections import defaultdict
import numpy as np
from brain_connectivity_vizuals import load_mesh_file, create_brain_connectivity_plot, create_modularity_visualization
import pandas as pd
import os
from netneurotools import plotting
from netneurotools.modularity import consensus_modularity

# Visualize States

In [5]:
# # CIMT_rest setup
# study = 'CIMT_rest'
# ics_path = '../CIMT_data/no_wm/rs_ICA_18c_hptf_ss/rs_ICA_18c_hptf_ss_agg__component_ica_denoised_.nii'
# atlas_path = '../CIMT_data/no_wm/rs_info/rs_01_study_specific_atlas_relabel_no_wm.nii'
# mesh_path = '../CIMT_data/no_wm/brain_filled_3_smoothed.gii'
# roi_coords_path = '../CIMT_data/no_wm/rs_info/resting_atlas_114_mapped_comma.csv'
# hmm_run = 'ICA_14c_no_TDE'
# optimal_num_states = 7
# covs = np.load(open(f'{study}/results/{hmm_run}/{optimal_num_states}_states/inf_params/covs_{optimal_num_states}_states.npy', 'rb'))
# print(covs.shape)

# state_plots_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/brain_state_plots'
# modularity_plots_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/modularity_plots'
# os.makedirs(state_plots_path, exist_ok=True)
# os.makedirs(modularity_plots_path, exist_ok=True)

In [7]:
# CIMT_task setup
study = 'CIMT_task'
ics_path = '../CIMT_data/no_wm/LLstim_ICA_18c_hptf_ss/LLstim_ICA_18c_hptf_ss_agg__component_ica_denoised_.nii'
atlas_path = '../CIMT_data/no_wm/LLstim_info/LLstim_01_study_specific_atlas_relabel_no_wm.nii'
mesh_path = '../CIMT_data/no_wm/brain_filled_3_smoothed.gii'
roi_coords_path = '../CIMT_data/no_wm/LLstim_info/atlas_114_mapped_comma.csv'
hmm_run = 'ICA_14c_no_TDE'
optimal_num_states = 4
covs = np.load(open(f'{study}/results/{hmm_run}/{optimal_num_states}_states/inf_params/covs_{optimal_num_states}_states.npy', 'rb'))
print(covs.shape)

state_plots_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/brain_state_plots'
modularity_plots_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/modularity_plots'
os.makedirs(state_plots_path, exist_ok=True)
os.makedirs(modularity_plots_path, exist_ok=True)

(4, 14, 14)


In [8]:
# # FPI setup
# study = 'FPI'
# ics_path = '../FPI_RS_share/no_wm/ICA_13c/ICA_13c_agg__component_ica_.nii'
# atlas_path = '../FPI_RS_share/no_wm/Template/01_study_specific_atlas_relabel_no_wm.nii'
# mesh_path = '../FPI_RS_share/full/Template/MDT_template0.gii'
# roi_coords_path = '../FPI_RS_share/no_wm/FPI_atlas_114_mapped_comma.csv'
# hmm_run = 'ICA_13c_no_TDE'
# optimal_num_states = 11
# covs = np.load(open(f'{study}/results/{hmm_run}/{optimal_num_states}_states/inf_params/covs_{optimal_num_states}_states.npy', 'rb'))
# print(covs.shape)

# state_plots_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/brain_state_plots'
# modularity_plots_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/modularity_plots'
# os.makedirs(state_plots_path, exist_ok=True)
# os.makedirs(mod_plots_path, exist_ok=True)

In [9]:
ics = nib.load(ics_path)
ics_data = ics.get_fdata()
ics_data.shape

(96, 96, 25, 14)

In [10]:
atlas = nib.load(atlas_path)
atlas_data = atlas.get_fdata()
atlas_data.shape

(96, 96, 25)

## Create n_ROI x n_IC Matrix

In [11]:
roi_to_ic = defaultdict(lambda: defaultdict(list)) # roi_idx: {ic: loadings for each coordinate of the roi}
for i in range(ics_data.shape[0]):
    for j in range(ics_data.shape[1]):
        for k in range(ics_data.shape[2]):
            for ic in range(ics_data.shape[3]):
                idx = int(atlas_data[i, j, k])
                if idx > 0:
                    roi_to_ic[idx][ic + 1].append(ics_data[i, j, k, ic])

In [12]:
roi_to_ic[1]

defaultdict(list,
            {1: [np.float64(-1.2676212787628174),
              np.float64(-1.4273436069488525),
              np.float64(-0.3156186640262604),
              np.float64(-0.36106473207473755),
              np.float64(-0.6865568161010742),
              np.float64(0.0560142882168293),
              np.float64(-1.29816734790802),
              np.float64(-1.6765402555465698),
              np.float64(-0.6822714805603027),
              np.float64(-0.35633111000061035),
              np.float64(-1.0258185863494873),
              np.float64(-1.537592887878418),
              np.float64(-0.507245659828186),
              np.float64(-0.39146071672439575),
              np.float64(-0.9879043698310852),
              np.float64(-1.4632269144058228),
              np.float64(-0.3816013038158417),
              np.float64(-0.4343782663345337),
              np.float64(-0.43342190980911255),
              np.float64(-0.5222105979919434),
              np.float64(-0.439111620187

In [13]:
A = np.zeros((len(roi_to_ic), ics_data.shape[3]))
for roi in sorted(roi_to_ic.keys()):
    for ic in sorted(roi_to_ic[roi].keys()):
        A[roi - 1, ic - 1] = np.mean(roi_to_ic[roi][ic])

A.shape

(114, 14)

## Create n_ROI x n_ROI Correlation Matrix

In [14]:
roi_corrs = []
for cov in covs:
    roi_cov = A @ cov @ A.T
    roi_cov = 0.5 * (roi_cov + roi_cov.T) # enforce symmetry just in case of floating point error
    
    std = np.sqrt(np.diag(roi_cov))
    std_safe = std.copy()
    std_safe[std_safe == 0] = 1
    
    denom = np.outer(std_safe, std_safe)
    roi_corr = roi_cov / denom
    
    std_zeros = (std == 0)
    roi_corr[std_zeros, :] = 0
    roi_corr[:, std_zeros] = 0
    roi_corrs.append(roi_corr)

np.array(roi_corrs).shape

(4, 114, 114)

## Make Plots

In [15]:
vertices, faces = load_mesh_file(mesh_path)

Loading mesh from: ../CIMT_data/no_wm/brain_filled_3_smoothed.gii
  Found faces array with shape: (18857, 3)
  Found vertices array with shape: (9510, 3)
  Successfully loaded mesh: 9510 vertices, 18857 faces


In [16]:
roi_coords_df = pd.read_csv(roi_coords_path)
roi_coords_df.head(20)

,mapped_index,original_index,roi_name,cog_x,cog_y,cog_z,cog_voxel_i,cog_voxel_j,cog_voxel_k
0,1,1,Acumbens_left,-15.355556,66.017901,-20.083333,79.284444,34.933333,142.814321
1,2,2,AID_left,-45.765769,64.444362,-7.542770,103.612615,44.965784,141.555489
2,3,3,AIP_left,-61.894410,29.911610,-21.016484,116.515528,34.186813,113.929288
3,4,4,AIV_left,-38.613395,69.366538,-9.036509,97.890716,43.770793,145.493230
4,5,5,Amygdala_left,-40.170219,15.355125,-37.531890,99.136175,20.974488,102.284100
5,6,8,Au1_left,-67.019196,-0.812283,4.944829,120.615357,54.955864,89.350173
6,7,9,AUD_left,-62.842693,3.178845,12.781300,117.274155,61.225040,92.543076
7,8,10,AuV_left,-69.197761,-2.039801,-6.915423,122.358209,45.467662,88.368159
8,9,11,Brainstem_left,-18.433435,-57.047145,-28.892489,81.746748,27.886009,44.362284
9,10,12,Cent_Gray_left,-7.648063,-22.936525,-0.421196,73.118450,50.663043,71.650780


In [17]:
# Generate brain state plots
roi_corrs_thr = []
for i in range(len(roi_corrs)):
    # Visualize top 2.5% of correlations for each state
    R = roi_corrs[i]
    iu = np.triu_indices_from(R, k=1)
    vals = np.abs(R[iu]) # (n_ROI * (n_ROI - 1) / 2,)
    thr = np.percentile(vals, 97.5)
    R_thr = np.zeros_like(R)
    keep = vals >= thr
    R_thr[iu[0][keep], iu[1][keep]] = R[iu[0][keep], iu[1][keep]]
    R_thr = R_thr + R_thr.T # symmetrize
    roi_corrs_thr.append(R_thr)
    create_brain_connectivity_plot(
        vertices=vertices, 
        faces=faces, 
        roi_coords_df=roi_coords_df, 
        connectivity_matrix=R_thr,
        plot_title=f"{study} HMM State {i + 1}",
        save_path=os.path.join(state_plots_path, f'state_{i + 1}.html'),
        node_size=8,
        node_color='purple',
        node_border_color='magenta',
        pos_edge_color='red',
        neg_edge_color='blue',
        edge_width=5,
        edge_threshold=0.0,
        mesh_color='lightgray',
        mesh_opacity=0.5,
        label_font_size=8,
        fast_render=False,
    )

Creating brain connectivity visualization: CIMT_task HMM State 1
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/brain_state_plots/state_1.html
Creating brain connectivity visualization: CIMT_task HMM State 2
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/brain_state_plots/state_2.html
Creating brain connectivity visualization: CIMT_task HMM State 3
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/brain_state_plots/state_3.html
Creating brain connectivity visualization: CIMT_task HMM State 4
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/brain_state_plots/state_4.html


In [18]:
np.savez(
    f'{study}/results/{hmm_run}/{optimal_num_states}_states/inf_params/all_corrs_{optimal_num_states}_states.npz', 
    centroids=np.array(roi_corrs),
)

In [19]:
# Generate modularity plots
modularity_res_path = f'{study}/results/{hmm_run}/{optimal_num_states}_states/inf_params/modularity_significance_analysis_omst_pos/results/k{optimal_num_states}_results'
for i in range(len(roi_corrs_thr)):
    modules = pd.read_csv(os.path.join(modularity_res_path, f'k{optimal_num_states}_state{i}_modules.csv'))
    create_modularity_visualization(
        vertices=vertices, 
        faces=faces, 
        roi_coords_df=roi_coords_df, 
        connectivity_matrix=roi_corrs_thr[i],
        # connectivity_matrix=roi_corrs[i],
        module_assignments=modules['module'],
        plot_title=f"{study} HMM State {i + 1} Modularity",
        save_path=os.path.join(modularity_plots_path, f'state_{i + 1}.html'),
        visualization_type="intra",  # "all", "intra", "inter", "nodes_only"
        node_size=6,
        edge_width=2,
        edge_color='red',  # Changed default to red
        mesh_color='lightgray',
        mesh_opacity=0.5,
        show_labels=True,
        label_font_size=8
    )

Creating modularity visualization: CIMT_task HMM State 1 Modularity
Active nodes after thresholding: 87 out of 114
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/modularity_plots/state_1.html
Creating modularity visualization: CIMT_task HMM State 2 Modularity
Active nodes after thresholding: 86 out of 114
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/modularity_plots/state_2.html
Creating modularity visualization: CIMT_task HMM State 3 Modularity
Active nodes after thresholding: 88 out of 114
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/modularity_plots/state_3.html
Creating modularity visualization: CIMT_task HMM State 4 Modularity
Active nodes after thresholding: 87 out of 114
Saved interactive visualization to: CIMT_task/results/ICA_14c_no_TDE/4_states/modularity_plots/state_4.html
